# Ablação da atenção entre amostras — Cenário 1 completo, DUAS T4 (Kaggle)

Pedido pós-qualificação (2026-11-09). Responde de forma definitiva se a atenção entre amostras
contribui, **refazendo a busca em grade** (os hiperparâmetros podem mudar sem a atenção).

**Cinco variantes, mesma máquina** (a seleção na grade depende da placa; não comparar com os números
publicados, que vieram de outra execução):

| variante | atenção entre amostras |
|---|---|
| `SAINTColnorm` | MISA ligada (publicado, refeito aqui) |
| `SAINTNoMISA` | MISA desligada |
| `FTTransformerCURRandom` | CUR (publicado, refeito aqui) |
| `FTTransformerCURRandomNoInter` | nenhuma |
| `FTTransformerCURRandomDense` | densa n×n, sem CUR |

**Protocolo publicado:** seleção por `GridSearchCV` (5 folds) com 200 épocas e paciência 16;
depois reajuste do vencedor com 200 épocas sem paciência (`refit_tf_nopatience.py`).
10 bases × 30 sementes.

**Prazo interno:** cada fase encerra sozinha ao atingir `DEADLINE_H` (os parciais já estão salvos e
espelhados em Output). **Não cancele a sessão** — o Kaggle só garante a saída quando ela termina sozinha.
Tudo é retomável (`RESUME_DIR`).

**Antes de rodar:** Settings → Accelerator → **GPU T4 x2**. Estimativa: ≈4–6 h de sessão.


In [ ]:
# ── 1. GPUs ──
import torch
print('CUDA:', torch.cuda.is_available(), '| placas:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(' ', i, torch.cuda.get_device_name(i))
assert torch.cuda.device_count() >= 2, 'Settings → Accelerator → GPU T4 x2 (com 1 placa, use as chamadas run_phase)'

In [ ]:
# ── 2. Repositório (branch com as correções) ──
import os, subprocess
GIT_URL     = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'
PROJECT_DIR = '/kaggle/working/sparse-lssvm-transformers-study'
BRANCH      = 'revisao/estatistica-e-proveniencia'
if os.path.exists(PROJECT_DIR):
    subprocess.run(['git', '-C', PROJECT_DIR, 'fetch', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, GIT_URL, PROJECT_DIR], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'checkout', BRANCH], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--rebase', 'origin', BRANCH], check=True)
os.chdir(PROJECT_DIR)
!git log --oneline -2
import sys; sys.path.insert(0, '.')
from src.tuning.grids import GRIDS
for v in ['SAINTNoMISA','FTTransformerCURRandomNoInter','FTTransformerCURRandomDense']:
    assert v in GRIDS, f'{v} ausente — refaça o pull'
print('variantes da ablação presentes')


In [ ]:
# ── 3. Dependências e dados (todos versionados no clone; nada é gerado em paralelo) ──
!pip install -q einops scikit-posthocs openpyxl 2>&1 | tail -n 1
from pathlib import Path
T1 = ['BCW','PID','HAB','VCP','GCR','AUS','AI4I','TWS','TWM','TWC']
falt = [d for d in T1 if not Path(f'data/raw/{d}.parquet').exists()]
assert not falt, f'sem cache: {falt} — refaça o pull'
from src.data.loaders import DatasetLoader
for ds in T1:
    X, y, _ = DatasetLoader.load(ds); print(f'  {ds:<5} N={len(y):>5} p={X.shape[1]}')
print('dados OK')


In [ ]:
# ── 4. Configuração, prazo interno e execução em duas placas ──
import json, os, shutil, time, subprocess, collections, statistics as st
from pathlib import Path
MODELS = ['SAINTColnorm','SAINTNoMISA','FTTransformerCURRandom',
          'FTTransformerCURRandomNoInter','FTTransformerCURRandomDense']
MODELS_STR = ' '.join(MODELS); T1_STR = ' '.join(T1)
DEADLINE_H = 10.5            # encerra as fases sozinho antes do limite de 12 h da sessão
T_START = time.time()
OUT = {'sel': 'results/ablinter_tier1_sel.json', 'refit': 'results/ablinter_tier1_refit.json'}
TOTAL = {'sel': len(MODELS)*10*30, 'refit': len(MODELS)*10*30}
Path('results').mkdir(exist_ok=True)

RESUME_DIR = None    # ex.: Path('/kaggle/input/ablinter-parcial') para retomar
if RESUME_DIR:
    for k, v in OUT.items():
        for suf in ('', '_g0', '_g1'):
            src = Path(RESUME_DIR) / Path(v).name.replace('.json', f'{suf}.json')
            if src.exists():
                shutil.copy(src, v.replace('.json', f'{suf}.json')); print('restaurado', src.name)

def save(key):
    shutil.copy(OUT[key], '/kaggle/working/' + Path(OUT[key]).name)
    print('salvo em Output:', Path(OUT[key]).name)

def _mirror(*paths):
    """Copia os JSONs em curso para /kaggle/working (raiz do Output).

    Os runners gravam em <repo>/results/ a cada execução concluída (append + escrita
    atômica), mas o painel Output do Kaggle mostra /kaggle/working: sem esta cópia
    periódica, uma sessão interrompida no meio de uma fase deixaria os dados só no
    diretório aninhado do clone. Espelhar a cada atualização de progresso torna os
    parciais sempre baixáveis (e reaproveitáveis via RESUME_DIR)."""
    for p in paths:
        try:
            if Path(p).exists(): shutil.copy(p, '/kaggle/working/' + Path(p).name)
        except Exception as e:
            print('  (aviso: falha ao espelhar', p, e, ')')

def _count(path):
    # Só erros ESPERADOS são tolerados (arquivo ainda não criado / meio de escrita).
    # Um except amplo aqui já esconderia um bug de import e o contador ficaria em 0
    # para sempre, com os processos rodando normalmente — foi o que aconteceu em
    # 2026-09-19 na primeira tentativa do SAINT.
    try: r = json.load(open(path))
    except (FileNotFoundError, ValueError): return 0, ''
    ok = [x for x in r if x.get('status', 'ok') == 'ok']
    if not ok: return 0, ''
    x = ok[-1]; f1 = x.get('test_f1_macro')
    info = f"último: {x.get('dataset', 'N=' + str(x.get('n')))}"
    info += f"/seed{x['seed']}" if 'seed' in x else ''
    info += f" F1={f1:.3f}" if isinstance(f1, (int, float)) else ''
    return len(ok), info

def run_phase(key, cmd, every=60, heartbeat=900):
    """Uma GPU."""
    out, total = OUT[key], TOTAL[key]; log = f'/kaggle/working/{key}.log'
    t0 = time.time(); last_n, last_print = -1, 0.0
    with open(log, 'w') as lf:
        p = subprocess.Popen(cmd, shell=True, stdout=lf, stderr=subprocess.STDOUT)
        while True:
            rc = p.poll(); n, info = _count(out); now = time.time()
            if n != last_n or rc is not None or now - last_print > heartbeat:
                el = (now - t0)/60; eta = el/n*(total-n) if n else float('nan')
                print(f'[{key}] {n:>4}/{total}  {100*n/total:3.0f}%  {el:6.0f} min  ETA {eta:5.0f} min  {info}', flush=True)
                _mirror(out)                     # parcial sempre baixável
                last_n, last_print = n, now
            if rc is not None: break
            time.sleep(every)
    print(f'[{key}] terminou (exit={rc}) — log: {log}'); save(key)
    if rc != 0: raise RuntimeError(f'{key}: exit {rc}\n' + ''.join(open(log).readlines()[-15:]))

def run_phase_2gpu(key, cmd_fmt, seeds=range(30), every=60, heartbeat=900):
    """Duas T4: um processo por placa, metade das sementes cada, shards separados.

    NÃO usar DataParallel/DDP: o FT-CUR e o SAINT fazem atenção inter-instâncias
    DENTRO do lote, então dividir o lote entre placas mudaria o modelo. Aqui o
    paralelismo é por experimento (sementes), que não altera nada.
    Dois processos gravando o mesmo JSON se sobrescrevem → um arquivo por placa.
    """
    seeds = list(seeds); half = len(seeds)//2
    base, total = OUT[key], TOTAL[key]
    shards, procs, logs, rcs = [], [], [], []
    for gpu, sds in [(0, seeds[:half]), (1, seeds[half:])]:
        shard = base.replace('.json', f'_g{gpu}.json'); shards.append(shard)
        log = f'/kaggle/working/{key}_g{gpu}.log'; logs.append(log)
        cmd = cmd_fmt.format(seeds=' '.join(map(str, sds)), output=shard)
        lf = open(log, 'w')
        # CUDA_VISIBLE_DEVICES prende o processo a uma placa; os limites de thread evitam
        # que os dois processos disputem os poucos vCPUs do Kaggle (cada um abriria tantas
        # threads de BLAS/OMP quantos núcleos houver, e o oversubscription chega a dobrar o
        # tempo do pré-processamento).
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu),
                   OMP_NUM_THREADS='2', MKL_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2',
                   NUMEXPR_NUM_THREADS='2', TOKENIZERS_PARALLELISM='false')
        procs.append(subprocess.Popen(cmd, shell=True, stdout=lf, stderr=subprocess.STDOUT, env=env))
        print(f'[{key}] GPU {gpu}: sementes {sds[0]}..{sds[-1]} -> {Path(shard).name}', flush=True)
    t0 = time.time(); last_n, last_print = -1, 0.0
    while True:
        rcs = [p.poll() for p in procs]
        n = sum(_count(s)[0] for s in shards); now = time.time()
        if n != last_n or all(rc is not None for rc in rcs) or now - last_print > heartbeat:
            el = (now - t0)/60; eta = el/n*(total-n) if n else float('nan')
            print(f'[{key}] {n:>4}/{total}  {100*n/total:3.0f}%  {el:6.0f} min  ETA {eta:5.0f} min', flush=True)
            _mirror(*shards)                     # parciais sempre baixáveis
            last_n, last_print = n, now
        if all(rc is not None for rc in rcs): break
        time.sleep(every)
    recs = []
    for s in shards:
        try: recs += json.load(open(s))
        except Exception: pass
    Path(base).write_text(json.dumps(recs, indent=1))
    print(f'[{key}] terminou (exits={rcs}) — {len(recs)} registros'); save(key)
    for s in shards: shutil.copy(s, '/kaggle/working/' + Path(s).name)
    if any(rc != 0 for rc in rcs): raise RuntimeError(f'{key}: exits {rcs} — ver {logs}')


def run_phase_2gpu(key, cmd_fmt, seeds=range(30), every=60, heartbeat=900):
    """Duas T4, um processo por placa (metade das sementes cada), shards separados.
    Encerra sozinho se o prazo interno DEADLINE_H estourar (os shards já estão gravados)."""
    seeds = list(seeds); half = len(seeds)//2
    base, total = OUT[key], TOTAL[key]
    shards, procs, logs = [], [], []
    for gpu, sds in [(0, seeds[:half]), (1, seeds[half:])]:
        shard = base.replace('.json', f'_g{gpu}.json'); shards.append(shard)
        log = f'/kaggle/working/{key}_g{gpu}.log'; logs.append(log)
        cmd = cmd_fmt.format(seeds=' '.join(map(str, sds)), output=shard)
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), OMP_NUM_THREADS='2',
                   MKL_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', NUMEXPR_NUM_THREADS='2')
        procs.append(subprocess.Popen(cmd, shell=True, stdout=open(log, 'w'),
                                      stderr=subprocess.STDOUT, env=env))
        print(f'[{key}] GPU {gpu}: sementes {sds[0]}..{sds[-1]} -> {Path(shard).name}', flush=True)
    t0 = time.time(); last_n, last_print, killed = -1, 0.0, False
    while True:
        rcs = [p.poll() for p in procs]
        n = sum(_count(s)[0] for s in shards); now = time.time()
        if (now - T_START)/3600 > DEADLINE_H and not killed:
            print(f'[{key}] PRAZO INTERNO ({DEADLINE_H} h) — encerrando os processos', flush=True)
            for p in procs:
                if p.poll() is None: p.terminate()
            killed = True
        if n != last_n or all(rc is not None for rc in rcs) or now - last_print > heartbeat:
            el = (now - t0)/60; eta = el/n*(total-n) if n else float('nan')
            print(f'[{key}] {n:>4}/{total}  {100*n/total:3.0f}%  {el:6.0f} min  ETA {eta:5.0f} min', flush=True)
            _mirror(*shards); last_n, last_print = n, now
        if all(rc is not None for rc in rcs): break
        time.sleep(every)
    recs = []
    for s in shards:
        try: recs += json.load(open(s))
        except (FileNotFoundError, ValueError): pass
    Path(base).write_text(json.dumps(recs, indent=1)); save(key)
    for s in shards:
        if Path(s).exists(): shutil.copy(s, '/kaggle/working/' + Path(s).name)
    print(f'[{key}] terminou (exits={rcs}) — {len(recs)} registros')
    for lg in logs:
        print(f'--- fim de {Path(lg).name} ---'); print(''.join(open(lg).readlines()[-5:]))
    return killed

print('config OK —', MODELS)


In [ ]:
# ── 5. Seleção: GridSearchCV com 200 épocas / paciência 16 (protocolo publicado) ──
killed = run_phase_2gpu('sel', "python -u scripts/run_tier1_gridcv.py --models " + MODELS_STR +
    " --datasets " + T1_STR + " --budget-epochs 200 --budget-patience 16"
    " --seeds {seeds} --output {output} --log-level WARNING")


In [ ]:
# ── 6. Reajuste do vencedor: 200 épocas sem paciência (protocolo publicado) ──
if killed:
    print('Seleção interrompida pelo prazo — reajuste não iniciado. Retome com RESUME_DIR.')
else:
    run_phase_2gpu('refit', "python -u scripts/refit_tf_nopatience.py --source " + OUT['sel'] +
        " --models " + MODELS_STR + " --seeds {seeds} --output {output}")


In [ ]:
# ── 7. Resumo rápido (F1-macro médio por variante) ──
import collections, statistics as st
for key in ('sel', 'refit'):
    try: R = [r for r in json.load(open(OUT[key])) if r.get('status') == 'ok']
    except FileNotFoundError: continue
    by = collections.defaultdict(list)
    for r in R: by[r['variant']].append(r['test_f1_macro'])
    print(key, {v: (len(x), round(st.mean(x), 4)) for v, x in by.items()})
